# Digital-Twin-Assisted Network Control for 6G Communication Using Real-Time Modeling and Predictive Optimization

Complete, self-contained implementation of the seven-stage framework (NumPy / SciPy / scikit-learn / Matplotlib only; **no GPU, no internet, no extra installs needed**).

| Stage | Content | Section |
|---|---|---|
| 1 | 6G network data collection (CSI, traffic, bandwidth utilisation, RSSI, energy, mobility) | §1 |
| 2 | Pre-processing: causal wavelet denoising, min-max, correlation-based selection, PCA | §2 |
| 3 | Real-time digital twin: multilayer graph model + synchronisation | §3 |
| 4 | AI prediction: LSTM, Transformer-style attention, AR time-series, ensemble | §4 |
| 5 | **QW-PNO** (Novelty 1): Dicke-constrained quantum-walk search + amplitude estimation + quantum-classical refinement | §6-7 |
| 6 | **KLR-DTC** (Novelty 2): Koopman/EDMD prediction + control-barrier functions + Age-of-Digital-Twin (AoDT) event-triggered sync | §8 |
| 7 | Physical network implementation (closed loop) | §10 |
| – | Comparison with QEDRL [16], QCDRL-LSTM [17], QSec-FL [18], QD-CE-A2C [19]; ROC / confusion matrices; ablation; robustness | §9-13 |

**Honest implementation notes**
* The 6G environment is a *simulator* (no public dataset is assumed). Replace `simulate_clean()` / `observe()` with your own CSI / traffic logs (same 33-column layout) to use real data.
* Quantum parts (Dicke state, continuous-time quantum walk on the Johnson graph, canonical amplitude estimation, quantum feature maps) are **exact classical state-vector simulations**.
* The four existing techniques are **compact surrogate re-implementations** built from their names/ideas (quantum-feature-map + RL / federated / actor-critic learners) because the authors' code is not available here. Swap in the official implementations if you have them, and state this in the paper.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import itertools, math, time, warnings
from scipy.spatial.distance import cdist
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_curve, roc_auc_score, confusion_matrix, accuracy_score,
                             f1_score, precision_score, recall_score)
warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

SEED = 42
rng = np.random.default_rng(SEED)

T_TOTAL, TRAIN_END, VAL_END = 3200, 1900, 2400      # train / val / test split (time ordered)
WIN = 10                                             # look-back window of the predictors
H_KOOP = 3                                           # Koopman look-ahead horizon (steps)
T0 = time.time()
print("setup ok")

## 1. Stage 1 – 6G network data collection (simulator)
33 collected features per second: traffic (6 BS), bandwidth utilisation (6), signal strength/RSSI (6), energy (6),
channel state information as sub-band SNR (8 bands), user mobility (1).
The 8 sub-bands mix sub-6 GHz / mmWave / sub-THz style bands (different bandwidth, SNR and hardware power).
Measurements carry Gaussian noise, outlier spikes and missing samples.

In [ ]:
N_BS, N_UE, N_BAND, K_ACT = 6, 24, 8, 4
BAND_BW   = np.array([20, 20, 40, 40, 80, 80, 160, 160], float)    # MHz
BAND_SNR0 = np.array([24, 22, 20, 18, 15, 13, 10, 8], float)       # mean SNR (dB) at reference PSD
BAND_CIRC = np.array([0.8, 0.8, 1.2, 1.2, 2.0, 2.0, 3.5, 3.5])     # circuit power (W)

FEATURE_NAMES = ([f"traffic_bs{i}" for i in range(N_BS)] + [f"bwutil_bs{i}" for i in range(N_BS)] +
                 [f"rssi_bs{i}" for i in range(N_BS)] + [f"energy_bs{i}" for i in range(N_BS)] +
                 [f"csi_band{j}" for j in range(N_BAND)] + ["mobility"])
GROUPS = dict(traffic=list(range(0, 6)), bwutil=list(range(6, 12)), rssi=list(range(12, 18)),
              energy=list(range(18, 24)), csi=list(range(24, 32)), mobility=[32])
NOISE_SD = np.concatenate([np.full(6, 3.0), np.full(6, 0.02), np.full(6, 1.5),
                           np.full(6, 1.5), np.full(8, 1.2), [1.0]])

def simulate_clean(T, rng):
    t = np.arange(T)
    # --- user mobility (m/s)
    ar_v = np.zeros(T)
    for i in range(1, T): ar_v[i] = 0.95 * ar_v[i-1] + rng.normal(0, 0.8)
    speed = np.clip(12 + 8*np.sin(2*np.pi*t/700) + ar_v, 1, 30)
    # --- traffic per BS (Mbps): diurnal + AR(1) + bursty surges
    base = rng.uniform(80, 125, N_BS); phi = rng.uniform(0, 2*np.pi, N_BS)
    diurnal = 1 + 0.35*np.sin(2*np.pi*t[:, None]/150 + phi[None, :])
    ar = np.zeros((T, N_BS))
    for i in range(1, T): ar[i] = 0.9*ar[i-1] + rng.normal(0, 4, N_BS)
    kern = np.array([(s+1)/5 if s < 5 else np.exp(-(s-5)/25) for s in range(150)])
    burst = np.zeros((T, N_BS))
    for t0 in np.where(rng.random(T) < 0.004)[0]:
        for b in rng.choice(N_BS, size=rng.integers(2, 4), replace=False):
            L = min(T - t0, len(kern)); burst[t0:t0+L, b] += rng.uniform(60, 160) * kern[:L]
    traffic = np.clip(base*diurnal + ar + burst, 20, 300)
    # --- sub-band channel (dB): mean + slow oscillation + mobility-dependent shadowing + fast fading
    rho = np.exp(-speed/200.0)
    common = np.zeros(T); indep = np.zeros((T, N_BAND))
    for i in range(1, T):
        common[i] = rho[i]*common[i-1] + np.sqrt(1-rho[i]**2)*rng.normal(0, 3.0)
        indep[i]  = rho[i]*indep[i-1]  + np.sqrt(1-rho[i]**2)*rng.normal(0, 3.5, N_BAND)
    Aj = rng.uniform(4.0, 7.0, N_BAND); Tj = rng.uniform(200, 500, N_BAND); pj = rng.uniform(0, 2*np.pi, N_BAND)
    osc = Aj*np.sin(2*np.pi*t[:, None]/Tj + pj)
    fade = rng.normal(0, 1.5, (T, N_BAND))
    snr = np.clip(BAND_SNR0 + osc + common[:, None] + indep + fade, 0, 35)
    bwutil = traffic/350.0
    off = rng.normal(0, 1.0, N_BS)
    rssi = -90 + 0.8*snr.mean(1)[:, None] + off[None, :]
    energy = 35 + 0.25*traffic
    return np.hstack([traffic, bwutil, rssi, energy, snr, speed[:, None]])

def observe(clean, rng, noise_mult=1.0, miss_rate=0.02, out_rate=0.005):
    obs = clean + rng.normal(0, 1, clean.shape)*NOISE_SD*noise_mult
    out = rng.random(clean.shape) < out_rate
    rr, cc = np.where(out)
    obs[rr, cc] += rng.choice([-1, 1], size=len(rr))*15*NOISE_SD[cc]*noise_mult
    obs[rng.random(clean.shape) < miss_rate] = np.nan
    return obs

clean = simulate_clean(T_TOTAL, rng)
obs   = observe(clean, rng)
D_true   = clean[:, GROUPS["traffic"]].sum(1)      # true aggregate demand (Mbps)
snr_true = clean[:, GROUPS["csi"]]                  # true instantaneous sub-band SNR (dB)
mob_true = clean[:, GROUPS["mobility"][0]]
print("data:", clean.shape, "| missing %.1f%%" % (100*np.isnan(obs).mean()),
      "| demand range %.0f-%.0f Mbps" % (D_true.min(), D_true.max()))

fig, ax = plt.subplots(1, 3, figsize=(15, 3.2))
ax[0].plot(obs[:600, 0], lw=.6, label="observed"); ax[0].plot(clean[:600, 0], lw=1.2, label="true"); ax[0].set_title("Traffic BS0 (Mbps)"); ax[0].legend()
ax[1].plot(obs[:600, 24], lw=.6); ax[1].plot(clean[:600, 24], lw=1.2); ax[1].set_title("CSI band 0 (dB)")
ax[2].plot(D_true[:T_TOTAL]); ax[2].axvline(TRAIN_END, c="k", ls="--"); ax[2].axvline(VAL_END, c="r", ls="--"); ax[2].set_title("Aggregate demand; train | val | test")
plt.tight_layout(); plt.show()

## 2. Stage 2 – Pre-processing and feature extraction
Everything is **causal** (usable in real time, no look-ahead): forward-fill imputation → Hampel outlier filter → sliding-window Haar wavelet denoising (soft universal threshold) → min-max normalisation → correlation-based feature selection → PCA (digital-twin state vector). All statistics are fitted on the training segment only.

In [ ]:
def causal_impute(X):
    X = X.copy(); T, F = X.shape
    for j in range(F):
        c = X[:, j]
        if np.isnan(c[0]): c[0] = c[~np.isnan(c)][0]          # start-up only
        idx = np.where(~np.isnan(c), np.arange(T), 0); idx = np.maximum.accumulate(idx)
        X[:, j] = c[idx]
    return X

def _haar_fwd(x): return (x[0::2]+x[1::2])/np.sqrt(2), (x[0::2]-x[1::2])/np.sqrt(2)
def _haar_inv(a, d):
    x = np.empty((a.shape[0]*2,)+a.shape[1:]); x[0::2] = (a+d)/np.sqrt(2); x[1::2] = (a-d)/np.sqrt(2); return x

def wavelet_denoise_window(W, levels=2, thr_scale=0.3):
    a = W; ds = []
    for _ in range(levels):
        a, d = _haar_fwd(a); ds.append(d)
    sigma = np.median(np.abs(ds[0]), axis=0)/0.6745 + 1e-9
    thr = thr_scale*sigma*np.sqrt(2*np.log(W.shape[0]))
    ds = [np.sign(d)*np.maximum(np.abs(d)-thr, 0) for d in ds]
    for d in reversed(ds): a = _haar_inv(a, d)
    return a[-1]

def causal_denoise(X, win=32, hampel_k=15.0, hampel_w=12):
    Xf = causal_impute(X); Xc = Xf.copy(); T, F = Xc.shape; out = np.zeros_like(Xc)
    for t in range(T):
        if t >= hampel_w:                       # Hampel reference = RAW history (so real level shifts are accepted)
            ref = Xf[t-hampel_w:t]; med = np.median(ref, 0); mad = 1.4826*np.median(np.abs(ref-med), 0) + 1e-9
            bad = np.abs(Xf[t]-med) > hampel_k*mad; Xc[t, bad] = med[bad]
        s = max(0, t-win+1); W = Xc[s:t+1]
        if len(W) < win: W = np.vstack([np.repeat(W[:1], win-len(W), 0), W])
        out[t] = wavelet_denoise_window(W)
    return out

class Preprocessor:
    def __init__(self, corr_thr=0.97, var_keep=0.99): self.corr_thr, self.var_keep = corr_thr, var_keep
    def fit(self, obs, train_end, protected):
        Xd = causal_denoise(obs[:train_end])
        self.lo = Xd.min(0); self.span = Xd.max(0) - self.lo + 1e-9
        Xn = (Xd - self.lo)/self.span
        C = np.corrcoef(Xn.T); keep, dropped = [], []
        for j in range(Xn.shape[1]):
            if j in protected or all(abs(C[j, i]) < self.corr_thr for i in keep): keep.append(j)
            else: dropped.append(j)
        self.keep, self.dropped = np.array(keep), dropped
        self.pos = {c: i for i, c in enumerate(self.keep)}
        self.pca = PCA(n_components=self.var_keep, svd_solver="full").fit(Xn[:, self.keep])
        self.k = self.pca.n_components_
        return self
    def transform(self, obs):
        Xd = causal_denoise(obs); Xn = (Xd - self.lo)/self.span
        return self.pca.transform(Xn[:, self.keep]), Xd
    def transform_clean(self, cl): return self.pca.transform(((cl - self.lo)/self.span)[:, self.keep])
    def phys(self, Z):
        Z = np.atleast_2d(Z); Xp = self.pca.inverse_transform(Z)*self.span[self.keep] + self.lo[self.keep]
        g = lambda name: Xp[:, [self.pos[c] for c in GROUPS[name]]]
        tr = np.clip(g("traffic"), 0, None)
        return dict(D=tr.sum(1), traffic=tr, snr=np.clip(g("csi"), 0, 35), mob=g("mobility")[:, 0], Xp=Xp)

PROTECTED = set(GROUPS["traffic"] + GROUPS["csi"] + GROUPS["mobility"])
pre = Preprocessor().fit(obs, TRAIN_END, PROTECTED)
Zobs, Xden = pre.transform(obs)
print("kept %d/33 features after correlation selection; dropped:" % len(pre.keep), [FEATURE_NAMES[j] for j in pre.dropped])
print("PCA components (99%% variance): %d" % pre.k)

te = slice(VAL_END, T_TOTAL)
raw_ff = causal_impute(obs)
rows = []
for g, idx in GROUPS.items():
    rows.append((g, np.sqrt(np.mean((raw_ff[te][:, idx]-clean[te][:, idx])**2)),
                 np.sqrt(np.mean((Xden[te][:, idx]-clean[te][:, idx])**2))))
print(pd.DataFrame(rows, columns=["group", "RMSE raw(ffill)", "RMSE denoised"]).round(3).to_string(index=False))

## 3. Stage 3 – Real-time digital-twin construction
The twin keeps a **multilayer graph** of the physical network (backhaul layer BS–BS, access layer UE–BS, interference layer BS–BS, resource layer BS–band) in one supra-adjacency matrix, and a graph-regularised state estimator `x̂ = (I + λL)⁻¹ y` (L = Laplacian of the backhaul layer) that fuses per-BS measurements over neighbouring base stations. The synchronisation logic (event-triggered, Age-of-Digital-Twin) is implemented in §8.

In [ ]:
class DigitalTwin:
    def __init__(self, rng, knn=3):
        self.bs_xy = rng.uniform(0, 1000, (N_BS, 2)); self.ue_xy = rng.uniform(0, 1000, (N_UE, 2))
        d_bb = cdist(self.bs_xy, self.bs_xy); d_ub = cdist(self.ue_xy, self.bs_xy)
        A_back = np.zeros((N_BS, N_BS))                        # layer 1: backhaul (k-NN, Gaussian weights)
        for i in range(N_BS):
            for j in np.argsort(d_bb[i])[1:knn+1]: A_back[i, j] = A_back[j, i] = np.exp(-(d_bb[i, j]/400)**2)
        A_intf = np.exp(-d_bb/300.0); np.fill_diagonal(A_intf, 0)   # layer 2: inter-cell interference
        A_acc = np.zeros((N_UE, N_BS)); A_acc[np.arange(N_UE), d_ub.argmin(1)] = 1   # layer 3: UE -> serving BS
        A_res = (rng.random((N_BS, N_BAND)) < 0.85).astype(float)                   # layer 4: BS <-> band availability
        self.layers = dict(backhaul=A_back, interference=A_intf, access=A_acc, resource=A_res)
        n = N_BS + N_UE + N_BAND
        S = np.zeros((n, n)); S[:N_BS, :N_BS] = A_back + 0.3*A_intf
        S[N_BS:N_BS+N_UE, :N_BS] = A_acc; S[:N_BS, N_BS:N_BS+N_UE] = A_acc.T
        S[:N_BS, N_BS+N_UE:] = A_res; S[N_BS+N_UE:, :N_BS] = A_res.T
        self.supra = S
        self.L = np.diag(A_back.sum(1)) - A_back
    def smooth(self, x_bs, lam=0.3):                           # x_bs: (T, N_BS)
        return np.linalg.solve(np.eye(N_BS) + lam*self.L, x_bs.T).T

twin_graph = DigitalTwin(np.random.default_rng(SEED+1))
print("supra-adjacency:", twin_graph.supra.shape, "| nodes: %d BS + %d UE + %d bands" % (N_BS, N_UE, N_BAND))
tr_raw = pre.phys(Zobs)["traffic"]; tr_true = clean[:, GROUPS["traffic"]]
for lam in [0.0, 0.02, 0.05, 0.3]:
    tr_hat = twin_graph.smooth(tr_raw, lam)
    print("graph-regularised per-BS traffic RMSE (test), lambda=%.2f: %.2f Mbps" % (lam, np.sqrt(np.mean((tr_hat[te]-tr_true[te])**2))))
print("(BS traffic is only weakly correlated across cells in this simulator, so heavy graph smoothing does not help;")
print(" the graph is kept as the twin's structural model and is exposed for real data with spatially correlated load.)")

## 4. Stage 4 – AI-based network behaviour prediction
Targets: next-step digital-twin state (PCA space) → traffic demand, channel (CSI) variation, mobility.
Models (all NumPy): **LSTM** (hand-written forward/BPTT, Adam, early stopping, gradient-checked), **Transformer-style multi-head self-attention** encoder (random projections + trained ridge read-out, positional encoding), **AR time-series** (ridge on lags) and persistence; the final predictor is a validation-weighted **ensemble**.
All models predict the *increment* `z[t+1] − z[t]`.

In [ ]:
def make_windows(Z, L=WIN, lo=0, hi=None):
    hi = len(Z)-1 if hi is None else hi
    idx = np.arange(max(L-1, lo), hi)
    X = np.stack([Z[i-L+1:i+1] for i in idx]); Y = Z[idx+1] - Z[idx]
    return idx, X, Y

class NumpyLSTM:
    def __init__(self, k, H=48, seed=0):
        r = np.random.default_rng(seed); s = 1/np.sqrt(H); self.H, self.k, self.ysc = H, k, 1.0
        self.p = dict(Wx=r.normal(0, s, (k, 4*H)), Wh=r.normal(0, s, (H, 4*H)), b=np.zeros(4*H),
                      Wy=r.normal(0, s, (H, k)), by=np.zeros(k))
        self.p["b"][H:2*H] = 1.0                                   # forget-gate bias
    @staticmethod
    def _sig(x): return 1/(1+np.exp(-np.clip(x, -30, 30)))
    def forward(self, X, keep=False):
        p, H = self.p, self.H; B, L, _ = X.shape
        h = np.zeros((B, H)); c = np.zeros((B, H)); cache = []
        for t in range(L):
            z = X[:, t]@p["Wx"] + h@p["Wh"] + p["b"]
            i = self._sig(z[:, :H]); f = self._sig(z[:, H:2*H]); g = np.tanh(z[:, 2*H:3*H]); o = self._sig(z[:, 3*H:])
            c_new = f*c + i*g; tc = np.tanh(c_new); h_new = o*tc
            if keep: cache.append((X[:, t], h, c, i, f, g, o, tc))
            h, c = h_new, c_new
        y = h@p["Wy"] + p["by"]
        return (y, cache, h) if keep else y
    def loss_grad(self, X, Y):
        p = self.p; y, cache, hL = self.forward(X, keep=True)
        err = y - Y; loss = np.mean(err**2); dy = 2*err/err.size
        g = {k: np.zeros_like(v) for k, v in p.items()}
        g["Wy"] = hL.T@dy; g["by"] = dy.sum(0)
        dh = dy@p["Wy"].T; dc = np.zeros_like(dh)
        for (xt, hp, cp, i, f, gg, o, tc) in reversed(cache):
            do = dh*tc; dc = dc + dh*o*(1-tc**2)
            di = dc*gg; dg = dc*i; df = dc*cp; dc_prev = dc*f
            dz = np.concatenate([di*i*(1-i), df*f*(1-f), dg*(1-gg**2), do*o*(1-o)], axis=1)
            g["Wx"] += xt.T@dz; g["Wh"] += hp.T@dz; g["b"] += dz.sum(0)
            dh = dz@p["Wh"].T; dc = dc_prev
        return loss, g
    def predict(self, X): return self.forward(X)*self.ysc
    def fit(self, Xtr, Ytr, Xva, Yva, epochs=60, bs=64, lr=3e-3, patience=10, seed=0, verbose=True):
        r = np.random.default_rng(seed); self.ysc = Ytr.std()
        m = {k: np.zeros_like(v) for k, v in self.p.items()}; v = {k: np.zeros_like(x) for k, x in self.p.items()}
        step, best, bad, hist = 0, (np.inf, None), 0, []
        for ep in range(epochs):
            perm = r.permutation(len(Xtr))
            for s in range(0, len(perm), bs):
                b = perm[s:s+bs]; _, g = self.loss_grad(Xtr[b], Ytr[b]/self.ysc)
                gn = np.sqrt(sum((x**2).sum() for x in g.values())); sc = min(1.0, 5.0/(gn+1e-9)); step += 1
                for k in self.p:
                    gk = g[k]*sc; m[k] = 0.9*m[k]+0.1*gk; v[k] = 0.999*v[k]+0.001*gk**2
                    self.p[k] -= lr*(m[k]/(1-0.9**step))/(np.sqrt(v[k]/(1-0.999**step))+1e-8)
            vl = np.mean((self.predict(Xva)-Yva)**2); hist.append(vl)
            if vl < best[0]-1e-9: best, bad = (vl, {k: x.copy() for k, x in self.p.items()}), 0
            else:
                bad += 1
                if bad >= patience: break
        self.p = best[1]; self.hist = hist
        if verbose: print("LSTM: %d epochs, best val MSE %.5f" % (len(hist), best[0]))
        return self

def lstm_gradcheck(k=5, H=6, B=4, L=5, eps=1e-6):
    r = np.random.default_rng(0); net = NumpyLSTM(k, H, seed=1)
    for key in net.p: net.p[key] += r.normal(0, 0.1, net.p[key].shape)
    X = r.normal(size=(B, L, k)); Y = r.normal(size=(B, k)); _, g = net.loss_grad(X, Y); worst = 0
    for key in net.p:
        for _ in range(4):
            idx = tuple(r.integers(0, s) for s in net.p[key].shape); old = net.p[key][idx]
            net.p[key][idx] = old+eps; lp, _ = net.loss_grad(X, Y); net.p[key][idx] = old-eps; lm, _ = net.loss_grad(X, Y)
            net.p[key][idx] = old; num = (lp-lm)/(2*eps)
            worst = max(worst, abs(num-g[key][idx])/(abs(num)+abs(g[key][idx])+1e-10))
    return worst
print("LSTM gradient check, max relative error: %.2e" % lstm_gradcheck())

class AttnReservoir:
    # Multi-head scaled-dot-product self-attention encoder (fixed random projections, sinusoidal positional encoding)
    # + closed-form ridge read-out.
    def __init__(self, k, L=WIN, heads=8, d=16, temp=2.0, alpha=10.0, seed=0):
        r = np.random.default_rng(seed); self.heads, self.d, self.alpha = heads, d, alpha
        self.Wq = r.normal(0, 1, (heads, k, d))/np.sqrt(k)*temp; self.Wk = r.normal(0, 1, (heads, k, d))/np.sqrt(k)*temp
        self.Wv = r.normal(0, 1, (heads, k, d))/np.sqrt(k)
        pos = np.arange(L)[:, None]; i = np.arange(k)[None, :]; ang = pos/np.power(10000, (2*(i//2))/k)
        self.pe = np.where(i % 2 == 0, np.sin(ang), np.cos(ang))*0.1
    def feats(self, X):
        Xp = X + self.pe[None]; fs = []
        for h in range(self.heads):
            Q = Xp[:, -1]@self.Wq[h]; K = Xp@self.Wk[h]; V = Xp@self.Wv[h]
            sc = np.einsum("bd,bld->bl", Q, K)/np.sqrt(self.d); sc -= sc.max(1, keepdims=True)
            a = np.exp(sc); a /= a.sum(1, keepdims=True); fs.append(np.einsum("bl,bld->bd", a, V))
        return np.concatenate(fs + [X[:, -1], X[:, -1]-X[:, -2], X[:, -1]-X[:, -3], X.mean(1)], axis=1)
    def fit(self, X, Y):
        F = self.feats(X); self.sc = StandardScaler().fit(F)
        self.rg = Ridge(alpha=self.alpha).fit(self.sc.transform(F), Y); return self
    def predict(self, X): return self.rg.predict(self.sc.transform(self.feats(X)))

class ARRidge:
    def __init__(self, alpha=1.0): self.alpha = alpha
    def fit(self, X, Y): self.m = Ridge(alpha=self.alpha).fit(X.reshape(len(X), -1), Y); return self
    def predict(self, X): return self.m.predict(X.reshape(len(X), -1))

K_PCA = pre.k
idx_tr, Xtr, Ytr = make_windows(Zobs, WIN, 0, TRAIN_END-1)
idx_va, Xva, Yva = make_windows(Zobs, WIN, TRAIN_END, VAL_END-1)
t0 = time.time()
lstm = NumpyLSTM(K_PCA, H=48, seed=0).fit(Xtr, Ytr, Xva, Yva)
best_a = min([1.0, 10.0, 100.0, 1000.0], key=lambda a: np.mean((AttnReservoir(K_PCA, alpha=a).fit(Xtr, Ytr).predict(Xva)-Yva)**2))
attn = AttnReservoir(K_PCA, alpha=best_a).fit(Xtr, Ytr)
arm  = ARRidge(alpha=1.0).fit(Xtr, Ytr)
mse = lambda m: np.mean((m.predict(Xva)-Yva)**2)
val_mse = dict(persistence=np.mean(Yva**2), AR=mse(arm), LSTM=mse(lstm), Attention=mse(attn))
w_l = 1/val_mse["LSTM"]; w_a = 1/val_mse["Attention"]; w_r = 1/val_mse["AR"]
W_ENS = np.array([w_l, w_a, w_r])/(w_l+w_a+w_r)
ens_predict = lambda X: W_ENS[0]*lstm.predict(X) + W_ENS[1]*attn.predict(X) + W_ENS[2]*arm.predict(X)
val_mse["Ensemble"] = np.mean((ens_predict(Xva)-Yva)**2)
print("validation MSE of next-step increment:", {k: round(v, 5) for k, v in val_mse.items()})
print("ensemble weights [LSTM, Attention, AR]:", W_ENS.round(3), "| training time %.1fs" % (time.time()-t0))

def predict_stream(pred_fn, Zs, L=WIN):
    idx = np.arange(L-1, len(Zs)); X = np.stack([Zs[i-L+1:i+1] for i in idx])
    Zh = Zs.copy(); Zh[idx] = Zs[idx] + pred_fn(X); return Zh

In [ ]:
def eval_pred(Zhat, idx):
    # compare the forecast made at t (for t+1) with the TRUE state at t+1
    ph = pre.phys(Zhat[idx]); Dt, St, Mt = D_true[idx+1], snr_true[idx+1], mob_true[idx+1]
    mape = np.mean(np.abs(ph["D"]-Dt)/Dt)*100
    r2 = 1 - np.sum((ph["D"]-Dt)**2)/np.sum((Dt-Dt.mean())**2)
    return dict(Demand_acc_pct=100-mape, Demand_R2=r2, Demand_MAE_Mbps=np.mean(np.abs(ph["D"]-Dt)),
                CSI_MAE_dB=np.mean(np.abs(ph["snr"]-St)), Mobility_MAE=np.mean(np.abs(ph["mob"]-Mt)))

idx_te = np.arange(VAL_END, T_TOTAL-1); idx_val = np.arange(TRAIN_END, VAL_END-1)
preds_obs = dict(Persistence=Zobs.copy(),
                 **{"AR (time-series)": predict_stream(arm.predict, Zobs)},
                 LSTM=predict_stream(lstm.predict, Zobs),
                 **{"Transformer-attention": predict_stream(attn.predict, Zobs)},
                 Ensemble=predict_stream(ens_predict, Zobs))
pred_table = pd.DataFrame({k: eval_pred(v, idx_te) for k, v in preds_obs.items()}).T
print(pred_table.round(3).to_string())

fig, ax = plt.subplots(1, 2, figsize=(13, 3.4))
ph = pre.phys(preds_obs["Ensemble"][idx_te])
ax[0].plot(D_true[idx_te+1][:250], label="true"); ax[0].plot(ph["D"][:250], label="ensemble forecast"); ax[0].set_title("Aggregate demand forecast (Mbps)"); ax[0].legend()
ax[1].plot(snr_true[idx_te+1][:250, 6], label="true"); ax[1].plot(ph["snr"][:250, 6], label="forecast"); ax[1].set_title("CSI forecast, band 6 (dB)"); ax[1].legend()
plt.tight_layout(); plt.show()

## 5. Network plant model (what the digital twin and the physical network evaluate)
A *configuration* activates exactly **k = 4 of n = 8 sub-bands** (Hamming-weight-4 bit string → 70 feasible configurations, this is what the Dicke-state encoding represents) and a *power factor* ∈ {0.5, 1, 2}. For a given (demand, sub-band SNR) state the model returns capacity/throughput, M/M/1 burst latency, outage-based reliability, finite-buffer packet-delivery ratio, energy, energy efficiency, a scalar cost **J** (lower is better) and a safety barrier **h** (h ≥ 0 ⇔ latency, reliability and power-budget constraints hold).

In [ ]:
CONFIGS = np.array([m for m in itertools.product([0, 1], repeat=N_BAND) if sum(m) == K_ACT], float)   # (70, 8)
N_CFG = len(CONFIGS)
PF = np.array([0.5, 1.0, 2.0])
ACT_CFG = np.repeat(np.arange(N_CFG), len(PF)); ACT_PF = np.tile(PF, N_CFG)                           # 210 actions
ACT_MASK = CONFIGS[ACT_CFG]; N_ACT = len(ACT_CFG)

R_SE, BURST, W_BASE, W_CAP, BUF = 0.5, 5e6, 1.0, 100.0, 20       # min spectral eff., burst size (bit), ms, ms, buffer
L_MAX, R_MIN, E_CAP = 40.0, 0.97, 24.0                           # QoS / safety limits: latency(ms), reliability, power(W)
W_J = dict(thr=0.35, lat=0.25, en=0.25, rel=0.15)
E_MAX = (CONFIGS*(BAND_CIRC + 2.0*PF.max()*BAND_BW/50.0)).sum(1).max()

def kpi_core(mask, pf, D, snr_db):
    # mask (...,8) pf (...) D (...) snr_db (...,8): broadcast-compatible arrays -> dict of arrays (...)
    sinr = 10**(snr_db/10)*pf[..., None]
    cap_j = BAND_BW*1e6*np.log2(1+sinr)*mask
    C = cap_j.sum(-1)/1e6                                                        # Mbps
    out = 1 - np.exp(-(2**R_SE-1)/sinr)
    rel = ((1-out)*cap_j).sum(-1)/(cap_j.sum(-1)+1e-9)
    E = (mask*(BAND_CIRC + 2.0*pf[..., None]*BAND_BW/50.0)).sum(-1)              # W
    mu = C*1e6/BURST; lam = D*1e6/BURST
    W = np.minimum(np.where(mu > lam, 1000.0/np.maximum(mu-lam, 1e-9) + W_BASE, W_CAP), W_CAP)
    rho = lam/np.maximum(mu, 1e-9); near = np.abs(rho-1) < 1e-6; rr = np.where(near, 0.5, rho)
    blk = np.clip(np.where(near, 1/(BUF+1), (1-rr)*rr**BUF/(1-rr**(BUF+1))), 0, 1)
    deliv = np.minimum(C, D)*rel
    J = (W_J["thr"]*(1-deliv/D) + W_J["lat"]*W/W_CAP + W_J["en"]*E/E_MAX + W_J["rel"]*np.minimum(1, (1-rel)/0.2))
    h = np.minimum(np.minimum(np.clip((L_MAX-W)/L_MAX, -1, 1), np.clip((rel-R_MIN)/(1-R_MIN), -1, 1)),
                   np.clip((E_CAP-E)/E_CAP, -1, 1))
    E = E + 0*C                                   # broadcast energy to the full (...) shape
    return dict(C=C, deliv=deliv, lat=W, rel=rel, pdr=rel*(1-blk), E=E, ee=deliv/E, J=J, h=h)

def kpi_grid(mask, pf, D, snr_db):      # actions x scenarios -> (A, S)
    return kpi_core(mask[:, None, :], pf[:, None], D[None, :], snr_db[None, :, :])
def kpi_pair(mask, pf, D, snr_db):      # one action per row
    return kpi_core(mask, pf, D, snr_db)

# Pre-compute the cost of every action for every (true) next state: used by the oracle, training of the RL baselines, labels
JTAB_ALL = kpi_grid(ACT_MASK, ACT_PF, D_true[1:], snr_true[1:])["J"].T          # (T-1, 210); row t = state at t+1
JSTAR = JTAB_ALL.min(1)
a_static = int(np.argmin(JTAB_ALL[:TRAIN_END].mean(0)))
k_ = kpi_pair(ACT_MASK[a_static][None], ACT_PF[a_static][None], D_true[VAL_END+1:VAL_END+2], snr_true[VAL_END+1:VAL_END+2])
print("actions: %d configs x %d power levels = %d | E_max %.1f W" % (N_CFG, len(PF), N_ACT, E_MAX))
print("oracle cost J* on test: mean %.3f | static best-on-train action: mean J %.3f" % (JSTAR[VAL_END:T_TOTAL-1].mean(), JTAB_ALL[VAL_END:, a_static].mean()))

## 6. Quantum building blocks (exact state-vector simulation)
* **Dicke state** |D⁸₄⟩ = uniform superposition of all weight-4 bit strings – the constrained encoding of *feasible* configurations (embedded and verified in the full 2⁸ space; the walk runs in the 70-dimensional constrained subspace).
* **Continuous-time quantum walk (CTQW) on the Johnson graph J(8,4)** (vertices = configurations, edges = swap one active band for one inactive band) with the spatial-search Hamiltonian H = −γA − Σ_{m∈marked}|m⟩⟨m|, γ = 1/λ_max(A).
* **Canonical quantum amplitude estimation (QAE)** with m ancilla qubits: exact outcome distribution P(y) = ½[F(y−Mθ) + F(y−M(1−θ))] (Fejér kernel), estimate p̂ = sin²(πy/M).

In [ ]:
def dicke_state(n, k):
    psi = np.zeros(2**n); idx = [i for i in range(2**n) if bin(i).count("1") == k]
    psi[idx] = 1/np.sqrt(len(idx)); return psi

idx_embed = [int("".join(str(int(b)) for b in c), 2) for c in CONFIGS]
psi_full = dicke_state(N_BAND, K_ACT)
assert np.isclose(np.linalg.norm(psi_full), 1) and np.allclose(psi_full[idx_embed], 1/np.sqrt(N_CFG)), "Dicke embedding failed"

HAM = (CONFIGS[:, None, :] != CONFIGS[None, :, :]).sum(-1)
A_JOHNSON = (HAM == 2).astype(float)                          # Johnson graph J(8,4): degree k(n-k) = 16
LAM1 = np.linalg.eigvalsh(A_JOHNSON).max(); GAMMA = 1.0/LAM1
PSI0 = np.ones(N_CFG)/np.sqrt(N_CFG)                          # = Dicke state in the constrained subspace
T_GRID = np.linspace(0.3, 14, 70)
print("Dicke state OK | Johnson graph degree %.0f, lambda_max %.0f, gamma %.4f" % (A_JOHNSON.sum(1)[0], LAM1, GAMMA))

def ctqw_search(marked):
    H = -GAMMA*A_JOHNSON; H[marked, marked] -= 1.0
    E, V = np.linalg.eigh(H); c0 = V.T@PSI0
    amp = V@(np.exp(-1j*np.outer(E, T_GRID))*c0[:, None])           # (70, nt)
    prob = np.abs(amp)**2; pm = prob[marked].sum(0); it = int(np.argmax(pm))
    return prob[:, it]/prob[:, it].sum(), float(pm[it]), float(T_GRID[it])

def qae_estimate(p, m_bits, rng):
    M = 2**m_bits; y = np.arange(M); p = np.atleast_1d(np.asarray(p, float)); est = np.zeros_like(p)
    def F(d):
        den = M*np.sin(np.pi*d/M); num = np.sin(np.pi*d)
        return np.where(np.abs(den) < 1e-10, 1.0, (num/np.where(np.abs(den) < 1e-10, 1, den))**2)
    for i, pi_ in enumerate(p):
        th = np.arcsin(np.sqrt(np.clip(pi_, 0, 1)))/np.pi
        P = 0.5*(F(y - M*th) + F(y - M*(1-th))); P = np.clip(P, 0, None); P /= P.sum()
        est[i] = np.sin(np.pi*rng.choice(y, p=P)/M)**2
    return est

# --- sanity tests of the quantum modules
rr_ = np.random.default_rng(0)
errs = [abs(qae_estimate(0.3, 6, rr_)[0]-0.3) for _ in range(300)]
print("QAE (m=6): mean abs error %.4f (theory ~ pi/M = %.3f)" % (np.mean(errs), np.pi/64))
succ = []
for _ in range(100):
    mk = rr_.choice(N_CFG, 7, replace=False); pr, pm, tt = ctqw_search(mk); succ.append(pm)
print("CTQW search, 7 marked of 70: mean success probability %.3f (uniform sampling = %.3f)" % (np.mean(succ), 7/70))

## 7. Stage 5 – QW-PNO: Quantum-Walk-based Predictive Network Optimisation  *(Novelty 1)*
For every decision epoch, from the **predicted** network state:
1. *Oracle*: cost of each feasible configuration (best power level) under the predicted state → the best M configurations are marked.
2. *Quantum-walk search* on J(8,4) starting from the Dicke state; the evolution time with maximum marked-probability is used.
3. *Amplitude estimation* estimates the walk's success probability (adaptively doubles M if it is too low) **and** each shortlisted candidate's probability of meeting the QoS/safety constraints under prediction uncertainty (scenarios sampled from the validation residuals).
4. *Quantum-classical refinement*: measure the walk (shots), take the K most frequent configurations, expand with the power levels, and rank them by a risk-aware score (½·mean + ½·CVaR₉₀ of J) subject to the QAE-estimated feasibility probability ≥ 0.8.

In [ ]:
SC = dict(S=48, K_short=8, M_mark=7, shots=64, m_qae=6, p_feas_min=0.8)

def draw_scenarios(resid, rng, S, scale=1.0):
    ii = rng.integers(0, len(resid), S); return scale*resid[ii, 0], scale*resid[ii, 1:]

def cvar(J, q=0.9):
    k = max(1, int(math.ceil((1-q)*J.shape[1]))); return np.sort(J, axis=1)[:, -k:].mean(1)

def clip_state(D, snr): return np.clip(D, 50, None), np.clip(snr, 0, 35)

def qw_pno(D_hat, snr_hat, resid, rng, use_qw=True, use_qae=True, risk=True, info=None):
    # 1. oracle on the predicted state (all feasible configurations, best power level)
    Jall = kpi_grid(ACT_MASK, ACT_PF, np.array([D_hat]), snr_hat[None, :])["J"][:, 0].reshape(N_CFG, len(PF))
    Jc = Jall.min(1)
    # 2.-3. quantum walk search + QAE on its success probability
    if use_qw:
        M = SC["M_mark"]
        for _ in range(2):
            marked = np.argsort(Jc)[:M]; prob, p_succ, t_star = ctqw_search(marked)
            p_hat = qae_estimate(p_succ, SC["m_qae"], rng)[0] if use_qae else p_succ
            if p_hat >= 0.4 or M >= 28: break
            M *= 2
        counts = np.bincount(rng.choice(N_CFG, SC["shots"], p=prob), minlength=N_CFG)
        order = np.lexsort((-prob, -counts)); short = order[:SC["K_short"]]
    else:                                                  # ablation: random candidate generation
        short = rng.choice(N_CFG, SC["K_short"], replace=False); p_hat, t_star, p_succ = np.nan, np.nan, np.nan
    # 4. refinement under uncertainty
    acts = np.array([c*len(PF) + j for c in short for j in range(len(PF))])
    if risk: rD, rS = draw_scenarios(resid, rng, SC["S"])
    else: rD, rS = np.zeros(1), np.zeros((1, N_BAND))
    Ds, Ss = clip_state(D_hat + rD, snr_hat[None, :] + rS)
    kp = kpi_grid(ACT_MASK[acts], ACT_PF[acts], Ds, Ss)
    score = 0.5*kp["J"].mean(1) + 0.5*cvar(kp["J"]); pfeas = (kp["h"] >= 0).mean(1)
    if use_qae and risk:
        pf_hat = qae_estimate(pfeas, SC["m_qae"], rng)
        key = np.where(pf_hat >= SC["p_feas_min"], score, 10 + (1-pf_hat) + 0.01*score)
    else: key = score
    ranked = acts[np.argsort(key)]
    if info is not None: info.update(p_succ=p_succ, t_star=t_star, short=short, hit_opt=int(np.argmin(Jc) in short))
    return ranked

# quick functional test on 200 test epochs
rt = np.random.default_rng(1); inf_ = []
resid_dummy = np.zeros((10, 1+N_BAND))
for t in range(VAL_END, VAL_END+200):
    d = {}; qw_pno(D_true[t+1], snr_true[t+1], resid_dummy, rt, info=d); inf_.append(d)
print("QW-PNO test (perfect prediction): walk success prob %.3f | true optimum configuration in shortlist: %.1f%%"
      % (np.nanmean([d["p_succ"] for d in inf_]), 100*np.mean([d["hit_opt"] for d in inf_])))

## 8. Stage 6 – KLR-DTC: Koopman-Lifted Risk-Aware Digital-Twin Control  *(Novelty 2)*
* **Koopman / EDMD**: the twin state z is lifted to ψ(z) = [1, z, z₁..ᵣ², RBF(z)] and a *linear* operator K is fitted by regularised least squares (ψ(zₜ₊₁) ≈ K ψ(zₜ)); multi-step forecasts are ψ·Kʰ – linear in the lifted space.
* **Control barrier functions**: each candidate is rolled out over the horizon inside the twin; the barrier h(x) = min(latency, reliability, power-budget margins). A candidate is accepted only if the (scenario-median) barrier stays ≥ 0 over the horizon **and** satisfies the discrete CBF condition h(xₖ₊₁) ≥ (1−γ)·h(xₖ). Among the accepted candidates the one with the lowest risk-aware (mean/CVaR) horizon cost is chosen; if none is safe, the candidate with the largest barrier is the fallback.
* **AoDT event-triggered synchronisation**: between synchronisations the twin runs open-loop on the Koopman model. The physical side compares its measurement with the twin's prediction and triggers a sync when the error exceeds a threshold **or** the Age-of-Digital-Twin (steps since last sync) exceeds A_max.

In [ ]:
class KoopmanEDMD:
    def __init__(self, k, n_rbf=40, r_sq=6, ridge=1e-3, seed=0):
        self.k, self.n_rbf, self.r, self.ridge, self.seed = k, n_rbf, min(r_sq, k), ridge, seed
    def lift(self, Z):
        Z = np.atleast_2d(Z); d2 = ((Z[:, None, :]-self.C[None])**2).sum(-1)
        return np.hstack([np.ones((len(Z), 1)), Z, Z[:, :self.r]**2, np.exp(-d2/(2*self.w**2))])
    def fit(self, Z):
        r = np.random.default_rng(self.seed); self.C = Z[r.choice(len(Z), self.n_rbf, replace=False)]
        dd = cdist(self.C, self.C); self.w = np.median(dd[dd > 0])
        P, Q = self.lift(Z[:-1]), self.lift(Z[1:])
        self.K = np.linalg.solve(P.T@P + self.ridge*np.eye(P.shape[1]), P.T@Q)
        Zl = np.hstack([Z[:-1]]); self.Klin = np.linalg.lstsq(Zl, Z[1:], rcond=None)[0]       # plain linear DMD for comparison
        return self
    def step(self, z): return (self.lift(z)@self.K)[0, 1:1+self.k]
    def rollout_all(self, Z, H):
        Psi = self.lift(Z); out = []
        for _ in range(H): Psi = Psi@self.K; out.append(Psi[:, 1:1+self.k].copy())
        return np.stack(out)

def _val_rmse(kp):
    R = kp.rollout_all(Zobs[:VAL_END], 3); n = VAL_END-TRAIN_END-3
    return sum(np.sqrt(np.mean((R[h][TRAIN_END:TRAIN_END+n]-Zobs[TRAIN_END+h+1:TRAIN_END+h+1+n])**2)) for h in range(3))
best_rg = min([1e-2, 1e-1, 1.0, 10.0], key=lambda g: _val_rmse(KoopmanEDMD(K_PCA, n_rbf=15, ridge=g).fit(Zobs[:TRAIN_END])))
koop = KoopmanEDMD(K_PCA, n_rbf=15, ridge=best_rg).fit(Zobs[:TRAIN_END])
print("Koopman ridge selected on validation:", best_rg)
Rk = koop.rollout_all(Zobs[:VAL_END], 5)
rows = []
for h in range(5):
    tgt = Zobs[TRAIN_END+h+1:VAL_END+h+1][:VAL_END-TRAIN_END-5]; sl = slice(TRAIN_END, TRAIN_END+len(tgt))
    kp_ = np.sqrt(np.mean((Rk[h][sl]-tgt)**2)); pers = np.sqrt(np.mean((Zobs[sl]-tgt)**2))
    Zl = Zobs[sl].copy()
    for _ in range(h+1): Zl = Zl@koop.Klin
    rows.append((h+1, pers, np.sqrt(np.mean((Zl-tgt)**2)), kp_))
print(pd.DataFrame(rows, columns=["horizon", "persistence RMSE", "linear DMD RMSE", "Koopman-EDMD RMSE"]).round(4).to_string(index=False))

# --- twin synchronisation policies
Zclean = pre.transform_clean(clean)
e_val = np.array([np.linalg.norm(koop.step(Zobs[t-1:t])-Zobs[t]) for t in range(TRAIN_END, VAL_END)])
TAU_E, A_MAX, PERIOD_N = float(np.percentile(e_val, 75)), 5, 5

def twin_stream(Zo, policy="event"):
    Zt = np.zeros_like(Zo); Zt[0] = Zo[0]; age = 0; n_sync = 0
    for t in range(1, len(Zo)):
        zp = koop.step(Zt[t-1:t]); age += 1
        if policy == "every": sync = True
        elif policy == "periodic": sync = age >= PERIOD_N
        else: sync = (np.linalg.norm(zp-Zo[t]) > TAU_E) or (age >= A_MAX)         # AoDT event trigger
        if sync: Zt[t] = Zo[t]; age = 0; n_sync += 1
        else: Zt[t] = zp
    return Zt, n_sync

def sync_error(Zt, sl=slice(VAL_END, T_TOTAL)):
    e = np.linalg.norm(Zt[sl]-Zclean[sl], axis=1)
    return np.sqrt(np.mean(e**2))/np.sqrt(np.mean(np.linalg.norm(Zclean[sl]-Zclean[sl].mean(0), axis=1)**2))

rows = []
for name, pol in [("Sync every step", "every"), ("Periodic (N=5)", "periodic"), ("AoDT event-triggered", "event")]:
    Zt, ns = twin_stream(Zobs, pol); rows.append((name, ns/T_TOTAL, sync_error(Zt)))
print(pd.DataFrame(rows, columns=["policy", "sync messages / step", "DT sync error (NRMSE)"]).round(4).to_string(index=False))

In [ ]:
def make_ctx(obs_in, sync="event", pred="ens"):
    Zo, _ = pre.transform(obs_in); Zs, ns = twin_stream(Zo, sync)
    Zhat = predict_stream(ens_predict, Zs) if pred == "ens" else Zs.copy()
    Zr = koop.rollout_all(Zs, H_KOOP); ph_h = pre.phys(Zhat); ph_c = pre.phys(Zs); ph_r = [pre.phys(Zr[h]) for h in range(H_KOOP)]
    Dk = [p["D"] for p in ph_r]; Sk = [p["snr"] for p in ph_r]
    Dk[0] = 0.5*(Dk[0]+ph_h["D"]); Sk[0] = 0.5*(Sk[0]+ph_h["snr"])             # h=1: fuse ensemble + Koopman forecast
    return dict(Zo=Zo, Zs=Zs, Zhat=Zhat, n_sync=ns, sync_rate=ns/len(Zo), Dh=ph_h["D"], Sh=ph_h["snr"],
                Dc=ph_c["D"], Sc=ph_c["snr"], Dk=Dk, Sk=Sk, sync_err=sync_error(Zs))

ctx_main = make_ctx(obs, "event", "ens")
ph_v = pre.phys(preds_obs["Ensemble"][idx_val])
RESID = np.column_stack([D_true[idx_val+1]-ph_v["D"], snr_true[idx_val+1]-ph_v["snr"]])      # validation residuals (D, 8 x SNR)
print("residual bank:", RESID.shape, "| demand residual std %.1f Mbps, CSI residual std %.2f dB" % (RESID[:, 0].std(), RESID[:, 1:].std()))

def klr_dtc(cands, ctx, t, resid, rng, h_prev, risk=True, gamma_c=0.6, S=32):
    scores = np.zeros(len(cands)); bars = []; ws = [0.5, 0.3, 0.2]
    for h in range(H_KOOP):
        if risk: rD, rS = draw_scenarios(resid, rng, S, scale=np.sqrt(h+1))
        else: rD, rS = np.zeros(1), np.zeros((1, N_BAND))
        Ds, Ss = clip_state(ctx["Dk"][h][t] + rD, ctx["Sk"][h][t][None, :] + rS)
        kp = kpi_grid(ACT_MASK[cands], ACT_PF[cands], Ds, Ss)
        scores += ws[h]*(0.5*kp["J"].mean(1) + 0.5*cvar(kp["J"])); bars.append(np.median(kp["h"], axis=1))
    bars = np.array(bars); bmin = bars.min(0); b1 = bars[0]
    cbf1 = (b1 >= (1-gamma_c)*h_prev) if h_prev > 0 else (b1 >= h_prev)               # discrete-time CBF condition
    ok = (bmin >= 0) & cbf1
    if ok.any(): j = np.where(ok)[0][np.argmin(scores[ok])]; fb = False
    else: j = int(np.argmax(bmin)); fb = True
    return cands[j], dict(n_ok=int(ok.sum()), fallback=fb)

def run_proposed(ctx, resid, use_qw=True, use_qae=True, use_cbf=True, risk=True, seed=0, idx=idx_te):
    r = np.random.default_rng(seed); acts = []; h_prev = 0.0; fb = 0
    for t in idx:
        ranked = qw_pno(max(ctx["Dh"][t], 50.0), np.clip(ctx["Sh"][t], 0, 35), resid, r, use_qw, use_qae, risk)
        if use_cbf: a, inf = klr_dtc(ranked, ctx, t, resid, r, h_prev, risk); fb += inf["fallback"]
        else: a = ranked[0]
        h_prev = float(kpi_pair(ACT_MASK[a][None], ACT_PF[a][None], ctx["Dc"][t:t+1], ctx["Sc"][t:t+1])["h"][0])
        acts.append(a)
    return np.array(acts)

def evaluate_actions(acts, idx=idx_te):
    kp = kpi_pair(ACT_MASK[acts], ACT_PF[acts], D_true[idx+1], snr_true[idx+1]); return kp

def summarize(kp, sync_rate=np.nan, pred_acc=np.nan, sync_err=np.nan):
    return {"Throughput (Mbps)": kp["deliv"].mean(), "Latency (ms)": kp["lat"].mean(), "Energy eff. (Mbps/W)": kp["ee"].mean(),
            "PDR": kp["pdr"].mean(), "Reliability": kp["rel"].mean(), "Power (W)": kp["E"].mean(), "Cost J": kp["J"].mean(),
            "QoS-violation rate": (kp["h"] < 0).mean(), "Pred. acc. (%)": pred_acc, "Sync msgs/step": sync_rate, "DT sync err": sync_err}

t0 = time.time(); a_prop = run_proposed(ctx_main, RESID); kp_prop = evaluate_actions(a_prop)
print("proposed controller on %d test epochs: %.1fs" % (len(idx_te), time.time()-t0))
print({k: round(float(v), 3) for k, v in summarize(kp_prop).items() if not np.isnan(v)})

## 9. Existing techniques for comparison – QEDRL [16], QCDRL-LSTM [17], QSec-FL [18], QD-CE-A2C [19]
Compact **surrogate re-implementations** on the *same* data, pre-processing, action space (210 configuration/power actions) and reward (−J measured on the true next state). All use a simulated 10-qubit **quantum feature map** (data re-uploading RY encoding + CNOT ring, ⟨Z⟩ and ⟨ZZ⟩ read-out, exact state-vector simulation):

| Method | Surrogate used here |
|---|---|
| QEDRL | quantum-feature linear Q-learning (ε-greedy) on the current observed state |
| QCDRL-LSTM | same learner, fed with the LSTM-predicted next state (quantum-classical hybrid with LSTM) |
| QSec-FL | federated linear Q-learning over 6 non-IID clients (one per BS view) with FedAvg and Gaussian-noise secure aggregation |
| QD-CE-A2C | quantum-feature dueling critic + softmax actor (advantage actor-critic) + cross-entropy elite-action regulariser |

Replace these by the authors' implementations if available.

In [ ]:
NQ, QL = 10, 2
_THETA = np.random.default_rng(7).uniform(-np.pi, np.pi, (QL, NQ))
_IDX = np.arange(2**NQ)
_ZS = 1 - 2*((_IDX[:, None] >> (NQ-1-np.arange(NQ)[None, :])) & 1)             # (1024, NQ) Pauli-Z eigenvalues
_ZZ = _ZS*np.roll(_ZS, -1, axis=1)
_PERM = [_IDX ^ (((_IDX >> (NQ-1-q)) & 1) << (NQ-1-((q+1) % NQ))) for q in range(NQ)]   # CNOT(q -> q+1)

def _ry(psi, ang, q):
    B = psi.shape[0]; p = psi.reshape(B, 2**q, 2, 2**(NQ-q-1)); c = np.cos(ang/2).reshape(-1, 1, 1); s = np.sin(ang/2).reshape(-1, 1, 1)
    out = np.empty_like(p); a0, a1 = p[:, :, 0, :], p[:, :, 1, :]
    out[:, :, 0, :] = c*a0 - s*a1; out[:, :, 1, :] = s*a0 + c*a1
    return out.reshape(B, -1)

def qfeatures(S01, chunk=600):
    out = []
    for s in range(0, len(S01), chunk):
        X = S01[s:s+chunk]; B = len(X); psi = np.zeros((B, 2**NQ)); psi[:, 0] = 1.0
        for l in range(QL):
            for q in range(NQ): psi = _ry(psi, np.pi*X[:, q]*(1+0.5*l) + _THETA[l, q], q)
            for q in range(NQ): psi = psi[:, _PERM[q]]
        pr = psi**2; out.append(np.hstack([pr@_ZS, pr@_ZZ, np.ones((B, 1))]))
    return np.vstack(out)

def state_vec(ph): return np.column_stack([ph["D"], ph["mob"], ph["snr"]])
_ph_tr = pre.phys(Zobs[:TRAIN_END]); _S_tr = state_vec(_ph_tr); ST_LO, ST_HI = _S_tr.min(0), _S_tr.max(0)
norm_state = lambda S: np.clip((S-ST_LO)/(ST_HI-ST_LO+1e-9), 0, 1)

# sanity: the simulated circuit is norm-preserving
_t = np.random.default_rng(0).random((3, NQ)); _psi = np.zeros((3, 2**NQ)); _psi[:, 0] = 1
for q in range(NQ): _psi = _ry(_psi, np.pi*_t[:, q], q)
assert np.allclose((_psi**2).sum(1), 1), "quantum circuit simulation not normalised"

def train_linear_q(Phi, Jt, steps=60000, alpha=0.05, seed=0, W0=None, eps0=1.0, eps1=0.05):
    r = np.random.default_rng(seed); W = np.zeros((Phi.shape[1], N_ACT)) if W0 is None else W0.copy()
    ii = r.integers(0, len(Phi), steps); rnd = r.random(steps); ra = r.integers(0, N_ACT, steps); eps = eps0; dec = (eps1/eps0)**(1/steps)
    for s in range(steps):
        phi = Phi[ii[s]]; a = ra[s] if rnd[s] < eps else int(np.argmax(phi@W))
        W[:, a] += alpha*(-Jt[ii[s], a] - phi@W[:, a])*phi; eps *= dec
    return W

def train_qsec_fl(S01_tr, Jt, n_clients=6, rounds=12, local_steps=3000, sigma_sec=0.002, seed=0):
    r = np.random.default_rng(seed); shards = np.array_split(np.arange(len(S01_tr)), n_clients)
    cl_phi = [qfeatures(np.clip(S01_tr[s] + r.normal(0, 0.03, S01_tr[s].shape), 0, 1)) for s in shards]
    W = np.zeros((cl_phi[0].shape[1], N_ACT))
    for rd in range(rounds):
        Ws = [train_linear_q(cl_phi[c], Jt[shards[c]], local_steps, 0.05, seed+10*rd+c, W, 1.0 if rd == 0 else 0.4, 0.05) for c in range(n_clients)]
        W = np.mean(Ws, 0) + r.normal(0, sigma_sec, W.shape)                  # secure aggregation with noise
    return W

def train_qd_ce_a2c(Phi, Jt, steps=60000, lr_a=0.05, lr_c=0.05, lr_ce=0.02, seed=0):
    r = np.random.default_rng(seed); F = Phi.shape[1]; Wp = np.zeros((F, N_ACT)); v = np.zeros(F); Wa = np.zeros((F, N_ACT)); ii = r.integers(0, len(Phi), steps)
    for s in range(steps):
        i = ii[s]; phi = Phi[i]; lg = phi@Wp*5.0; pi = np.exp(lg-lg.max()); pi /= pi.sum()
        a = r.choice(N_ACT, p=pi); rew = -Jt[i, a]
        Aall = phi@Wa; Qall = phi@v + Aall - Aall.mean(); err = rew - Qall[a]          # dueling critic Q = V + A - mean(A)
        v += lr_c*err*phi; Wa[:, a] += lr_c*err*phi; Wa -= lr_c*err*phi[:, None]/N_ACT
        adv = rew - phi@v; oh = np.zeros(N_ACT); oh[a] = 1
        Wp += lr_a*adv*np.outer(phi, oh-pi)                                              # advantage actor update
        oh2 = np.zeros(N_ACT); oh2[int(np.argmax(Qall))] = 1; Wp += lr_ce*np.outer(phi, oh2-pi)   # cross-entropy elite action
    return Wp

# ---------------- training on the training segment
tr_i = np.arange(WIN, TRAIN_END-1); Jt_tr = JTAB_ALL[tr_i]
ph_obs_all = pre.phys(Zobs); ph_lstm_all = pre.phys(preds_obs["LSTM"])
S01_obs = norm_state(state_vec(ph_obs_all)); S01_lstm = norm_state(state_vec(ph_lstm_all))
t0 = time.time()
PHI_obs = qfeatures(S01_obs); PHI_lstm = qfeatures(S01_lstm)
print("quantum features computed: %s in %.1fs" % (PHI_obs.shape, time.time()-t0))
t0 = time.time()
W_qedrl  = train_linear_q(PHI_obs[tr_i], Jt_tr, seed=1)
W_qcdrl  = train_linear_q(PHI_lstm[tr_i], Jt_tr, seed=2)
W_qsecfl = train_qsec_fl(S01_obs[tr_i], Jt_tr, seed=3)
W_qdce   = train_qd_ce_a2c(PHI_obs[tr_i], Jt_tr, seed=4)
print("baselines trained in %.1fs" % (time.time()-t0))

def baseline_actions(Zo, idx):
    S_o = norm_state(state_vec(pre.phys(Zo[idx]))); Zl = predict_stream(lstm.predict, Zo); S_l = norm_state(state_vec(pre.phys(Zl[idx])))
    Po, Pl = qfeatures(S_o), qfeatures(S_l)
    return {"QEDRL [16]": np.argmax(Po@W_qedrl, 1), "QCDRL-LSTM [17]": np.argmax(Pl@W_qcdrl, 1),
            "QSec-FL [18]": np.argmax(Po@W_qsecfl, 1), "QD-CE-A2C [19]": np.argmax(Po@W_qdce, 1)}

## 10. Stage 7 – Physical-network implementation: closed-loop comparison on the test segment
At each decision epoch the controller sees only past observations, selects (configuration, power) for the next interval, and the physical network (true state at t+1) returns the KPIs. The proposed method is repeated with 3 random seeds (shots / QAE / scenario sampling are stochastic).

In [ ]:
t0 = time.time()
res = {}; sync_main = ctx_main["sync_rate"]
prop_runs = [evaluate_actions(run_proposed(ctx_main, RESID, seed=s)) for s in range(3)]
pa_prop = eval_pred(ctx_main["Zhat"], idx_te)["Demand_acc_pct"]
rows_prop = pd.DataFrame([summarize(k, sync_main, pa_prop, ctx_main["sync_err"]) for k in prop_runs])
res["Proposed (QW-PNO + KLR-DTC)"] = rows_prop.mean().to_dict(); prop_std = rows_prop.std()
base_act = baseline_actions(Zobs, idx_te)
pa_pers = eval_pred(preds_obs["Persistence"], idx_te)["Demand_acc_pct"]; pa_lstm = eval_pred(preds_obs["LSTM"], idx_te)["Demand_acc_pct"]
for name, a in base_act.items():
    res[name] = summarize(evaluate_actions(a), 1.0, pa_lstm if "LSTM" in name else pa_pers, np.nan)
res["Static (best on train)"] = summarize(evaluate_actions(np.full(len(idx_te), a_static)))
res["Oracle (clairvoyant bound)"] = summarize(evaluate_actions(JTAB_ALL[idx_te].argmin(1)))
main_table = pd.DataFrame(res).T
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
print(main_table.round(4).to_string())
print("\nProposed, std over 3 controller seeds:", prop_std[["Throughput (Mbps)", "Latency (ms)", "Cost J", "QoS-violation rate"]].round(4).to_dict())
print("elapsed %.1fs" % (time.time()-t0))

fig, ax = plt.subplots(2, 3, figsize=(16, 7)); cols = ["Throughput (Mbps)", "Latency (ms)", "Energy eff. (Mbps/W)", "PDR", "Reliability", "QoS-violation rate"]
order = [n for n in main_table.index if "Oracle" not in n]; colors = ["#d62728"] + ["#7f7f7f"]*4 + ["#bcbd22"]
for a_, c in zip(ax.ravel(), cols):
    a_.bar(range(len(order)), main_table.loc[order, c].values, color=colors); a_.set_xticks(range(len(order))); a_.set_xticklabels([o.split(" (")[0] for o in order], rotation=40, ha="right", fontsize=8); a_.set_title(c)
    if c in ("PDR", "Reliability"): a_.set_ylim(main_table.loc[order, c].min()*0.99, 1.0)
plt.tight_layout(); plt.show()

## 11. Network-stress prediction: ROC curves and confusion matrices
Binary task: will the network be in a *high-stress regime* at t+1 (best achievable cost J* above the 75th percentile of the training segment)? Each method predicts with the features of its own pipeline (proposed: twin forecasts of the ensemble + Koopman horizon states; baselines: their quantum features of the current / LSTM-predicted state; QSec-FL: federated logistic regression with secure-aggregation noise; QD-CE-A2C: MLP). Classifiers are fitted on the **validation** segment, the decision threshold is Youden's J on validation, and everything is reported on the **test** segment.

In [ ]:
stress_thr = np.percentile(JSTAR[:TRAIN_END], 75)
y_val = (JSTAR[idx_val] > stress_thr).astype(int); y_te = (JSTAR[idx_te] > stress_thr).astype(int)
print("stress prevalence: val %.2f | test %.2f" % (y_val.mean(), y_te.mean()))

def prop_feats(ctx): return np.column_stack([ctx["Dh"], ctx["Sh"], ctx["Dk"][1], ctx["Dk"][2], ctx["Dc"]])
F_prop = prop_feats(ctx_main)
sc_p = StandardScaler().fit(F_prop[idx_val]); clf_p = LogisticRegression(C=1.0, max_iter=2000).fit(sc_p.transform(F_prop[idx_val]), y_val)
score = {"Proposed (DT forecast)": (clf_p.predict_proba(sc_p.transform(F_prop[idx_val]))[:, 1], clf_p.predict_proba(sc_p.transform(F_prop[idx_te]))[:, 1])}

def fit_lr(P):
    sc = StandardScaler().fit(P[idx_val]); m = LogisticRegression(C=1.0, max_iter=2000).fit(sc.transform(P[idx_val]), y_val)
    return m.predict_proba(sc.transform(P[idx_val]))[:, 1], m.predict_proba(sc.transform(P[idx_te]))[:, 1]
score["QEDRL [16]"] = fit_lr(PHI_obs[:, :-1]); score["QCDRL-LSTM [17]"] = fit_lr(PHI_lstm[:, :-1])

def fed_logreg(Xs, ys, rounds=15, epochs=25, lr=0.5, sigma=0.002, seed=0):
    r = np.random.default_rng(seed); w = np.zeros(Xs[0].shape[1])
    for _ in range(rounds):
        ws = []
        for X, y in zip(Xs, ys):
            wl = w.copy()
            for _ in range(epochs): wl -= lr*X.T@(1/(1+np.exp(-X@wl)) - y)/len(y)
            ws.append(wl)
        w = np.mean(ws, 0) + r.normal(0, sigma, w.shape)
    return w
rs = np.random.default_rng(5); sh = np.array_split(np.arange(len(idx_val)), 6)
Xc = [qfeatures(np.clip(S01_obs[idx_val][s] + rs.normal(0, 0.03, S01_obs[idx_val][s].shape), 0, 1)) for s in sh]
w_fl = fed_logreg(Xc, [y_val[s] for s in sh]); sg = lambda z: 1/(1+np.exp(-z))
score["QSec-FL [18]"] = (sg(PHI_obs[idx_val]@w_fl), sg(PHI_obs[idx_te]@w_fl))
mlp = MLPClassifier((32,), max_iter=800, random_state=0, alpha=1e-3).fit(PHI_obs[idx_val][:, :-1], y_val)
score["QD-CE-A2C [19]"] = (mlp.predict_proba(PHI_obs[idx_val][:, :-1])[:, 1], mlp.predict_proba(PHI_obs[idx_te][:, :-1])[:, 1])

rows, cms = [], {}
fig, ax = plt.subplots(1, 1, figsize=(5.8, 5))
for name, (sv, st) in score.items():
    fpr, tpr, thr = roc_curve(y_val, sv); th = thr[np.argmax(tpr-fpr)]; yp = (st >= th).astype(int)
    rows.append((name, roc_auc_score(y_te, st), accuracy_score(y_te, yp), precision_score(y_te, yp, zero_division=0), recall_score(y_te, yp), f1_score(y_te, yp)))
    cms[name] = confusion_matrix(y_te, yp); f, t_, _ = roc_curve(y_te, st); ax.plot(f, t_, lw=2 if "Proposed" in name else 1.2, label="%s (AUC %.3f)" % (name, rows[-1][1]))
ax.plot([0, 1], [0, 1], "k--", lw=.8); ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate"); ax.set_title("ROC – network-stress prediction (test)"); ax.legend(fontsize=7, loc="lower right")
plt.tight_layout(); plt.show()
roc_table = pd.DataFrame(rows, columns=["Method", "AUC", "Accuracy", "Precision", "Recall", "F1"]).set_index("Method"); print(roc_table.round(4).to_string())

fig, ax = plt.subplots(1, len(cms), figsize=(3.2*len(cms), 3.2))
for a_, (name, cm) in zip(ax, cms.items()):
    a_.imshow(cm, cmap="Blues"); a_.set_title(name, fontsize=8); a_.set_xticks([0, 1]); a_.set_yticks([0, 1]); a_.set_xlabel("predicted"); a_.set_ylabel("true"); a_.grid(False)
    for i in range(2):
        for j in range(2): a_.text(j, i, cm[i, j], ha="center", va="center", color="black" if cm[i, j] < cm.max()/2 else "white")
plt.tight_layout(); plt.show()

## 12. Ablation study
Each row removes exactly one component of the proposed framework (the rest unchanged).

In [ ]:
t0 = time.time()
ctx_per = make_ctx(obs, "periodic", "ens"); ctx_evy = make_ctx(obs, "every", "ens"); ctx_pers = make_ctx(obs, "event", "persist")
ABL = [("Proposed (full)",                       ctx_main, dict()),
       ("w/o QW-PNO (random candidates)",        ctx_main, dict(use_qw=False)),
       ("w/o QAE (no feasibility filter)",       ctx_main, dict(use_qae=False)),
       ("w/o risk-aware scenarios (mean cost)",  ctx_main, dict(risk=False)),
       ("w/o KLR-DTC (no Koopman horizon / CBF)", ctx_main, dict(use_cbf=False)),
       ("w/o AoDT trigger (periodic sync, N=5)", ctx_per,  dict()),
       ("w/o AI prediction (reactive twin)",     ctx_pers, dict()),
       ("Full + sync every step (comm. cost x2.5)", ctx_evy, dict())]
rows = {}
for name, cx, kw in ABL:
    kp = evaluate_actions(run_proposed(cx, RESID, seed=0, **kw)); rows[name] = summarize(kp, cx["sync_rate"], np.nan, cx["sync_err"])
abl_table = pd.DataFrame(rows).T[["Throughput (Mbps)", "Latency (ms)", "Energy eff. (Mbps/W)", "PDR", "Reliability", "Cost J", "QoS-violation rate", "Sync msgs/step", "DT sync err"]]
print(abl_table.round(4).to_string()); print("elapsed %.1fs" % (time.time()-t0))
fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
ax[0].barh(abl_table.index[::-1], abl_table["Cost J"].values[::-1], color="#1f77b4"); ax[0].set_title("Cost J (lower is better)")
ax[1].barh(abl_table.index[::-1], 100*abl_table["QoS-violation rate"].values[::-1], color="#d62728"); ax[1].set_title("QoS-violation rate (%)")
plt.tight_layout(); plt.show()

## 13. Robustness testing
Models are trained once on clean-regime data; at test time the measurement noise is scaled (×1 … ×4), the missing-sample rate is raised, and glitch outliers are increased. All methods see the same perturbed observations through the same causal pre-processing.

In [ ]:
t0 = time.time(); r2 = np.random.default_rng(99); start = VAL_END - 64
SETTINGS = [("nominal", 1, 0.02), ("noise x2", 2, 0.02), ("noise x3", 3, 0.05), ("noise x4 + 10% missing", 4, 0.10)]
rob = []
for lab, nm, mr in SETTINGS:
    ob = obs.copy(); ob[start:] = observe(clean[start:], r2, noise_mult=nm, miss_rate=mr, out_rate=0.005*nm)
    cx = make_ctx(ob, "event", "ens"); kp = evaluate_actions(run_proposed(cx, RESID, seed=0))
    rob.append((lab, "Proposed", kp["J"].mean(), (kp["h"] < 0).mean(), kp["lat"].mean(), eval_pred(cx["Zhat"], idx_te)["Demand_acc_pct"]))
    Zo = cx["Zo"]
    for name, a in baseline_actions(Zo, idx_te).items():
        k2 = evaluate_actions(a); rob.append((lab, name, k2["J"].mean(), (k2["h"] < 0).mean(), k2["lat"].mean(), np.nan))
rob = pd.DataFrame(rob, columns=["setting", "method", "J", "violation", "latency", "pred_acc"])
print(rob.pivot(index="method", columns="setting", values="J")[[s[0] for s in SETTINGS]].round(4).to_string())
print(); print(rob.pivot(index="method", columns="setting", values="violation")[[s[0] for s in SETTINGS]].round(4).to_string())
print("proposed forecast accuracy (%):", rob[rob.method == "Proposed"].set_index("setting")["pred_acc"].round(2).to_dict()); print("elapsed %.1fs" % (time.time()-t0))
fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
for m_, g in rob.groupby("method"):
    g = g.set_index("setting").loc[[s[0] for s in SETTINGS]]
    ax[0].plot(range(len(SETTINGS)), g["J"].values, marker="o", lw=2.5 if m_ == "Proposed" else 1.2, label=m_)
    ax[1].plot(range(len(SETTINGS)), 100*g["violation"].values, marker="o", lw=2.5 if m_ == "Proposed" else 1.2, label=m_)
for a_, t_ in zip(ax, ["Cost J vs perturbation", "QoS-violation rate (%) vs perturbation"]):
    a_.set_xticks(range(len(SETTINGS))); a_.set_xticklabels([s[0] for s in SETTINGS], rotation=15); a_.set_title(t_)
ax[0].legend(fontsize=7); plt.tight_layout(); plt.show()

## 14. Final summary and export

In [ ]:
import os
os.makedirs("results_dt6g", exist_ok=True)
main_table.round(5).to_csv("results_dt6g/comparison.csv"); abl_table.round(5).to_csv("results_dt6g/ablation.csv")
roc_table.round(5).to_csv("results_dt6g/roc_metrics.csv"); rob.round(5).to_csv("results_dt6g/robustness.csv"); pred_table.round(5).to_csv("results_dt6g/prediction_models.csv")
best = main_table.drop(index=["Oracle (clairvoyant bound)"])["Cost J"].idxmin()
print("Lowest-cost method on the test segment:", best)
P = main_table.loc["Proposed (QW-PNO + KLR-DTC)"]; B = main_table.drop(index=["Proposed (QW-PNO + KLR-DTC)", "Static (best on train)", "Oracle (clairvoyant bound)"])
bb = B["Cost J"].idxmin()
print("Proposed vs best surrogate baseline (%s): J %.4f vs %.4f | QoS violations %.1f%% vs %.1f%% | latency %.1f vs %.1f ms"
      % (bb, P["Cost J"], B.loc[bb, "Cost J"], 100*P["QoS-violation rate"], 100*B.loc[bb, "QoS-violation rate"], P["Latency (ms)"], B.loc[bb, "Latency (ms)"]))
print("Results saved in ./results_dt6g/ ; total notebook runtime %.1f min" % ((time.time()-T0)/60))

### Notes for the paper / how to adapt
* **Data**: to use real traces keep the 33-column layout (`FEATURE_NAMES`) and replace `clean`/`obs`; `D_true`, `snr_true` must hold the realised demand and sub-band SNR used by the plant model (`kpi_core`).
* **Plant model constants** (`L_MAX`, `R_MIN`, `E_CAP`, `W_J`, band table) are the only place where the 6G system assumptions live.
* **Quantum parts** run on a classical simulator, so no quantum speed-up is claimed; what is demonstrated is the algorithmic pipeline (Dicke-constrained encoding → walk search → QAE → classical refinement).
* **Baselines [16]–[19]** are surrogates (see §9); report them as "re-implemented under identical conditions" or substitute the official code.
* Everything is seeded (`SEED = 42`), results vary slightly only through the stochastic controller (shots, QAE, scenario sampling).